# ScreamingFace quickstart

Six steps: inspect the public Leaderboards, connect a provider, run a Benchmark, read the
Report, publish its Candidate Result, and replay its URL4. The wider interface is covered in
`01_client_tour.ipynb`.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [1]:
import screamingface as sf

sf.configure(
    engine_url="http://127.0.0.1:9108",
    scoreboard_url="http://127.0.0.1:9106",
)

BENCHMARK_ID = "draco/smoke"

## 1 · Leaderboards

Leaderboard discovery reads from the Scoreboard and does not require a provider connection.
`just stack-up` registers the local `draco/smoke` development Leaderboard, so discovery,
evaluation, and publication use the same Benchmark id. Both values render as interactive,
brand-system notebook widgets.

In [2]:
leaderboards = sf.leaderboards.list()
leaderboards

Leaderboards(1)

In [3]:
leaderboard = sf.leaderboards.get(BENCHMARK_ID, top=10)
leaderboard

Leaderboard('draco/smoke', entries=1, baselines=0)

## 2 · Connect

`sf.connect()` renders the Engine-backed provider panel. A key entered here goes to the SF
Engine for AI Gateway validation and encrypted storage; the notebook never retains it. On a
hosted Engine the panel asks for Cloudflare Access login first.

In [4]:
sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efe…

## 3 · Evaluate

`draco/smoke` keeps DRACO's execution structure but reduces it to one pinned Case, one
criterion, and one Judge pass. It makes two paid calls — one Candidate answer, one Judge
grade — so the score is diagnostic and **not comparable** to canonical DRACO.

Running this cell makes those two inexpensive calls. While it runs, the live panel shows
progress, model calls, tokens and cost.

In [5]:
candidate = sf.Model("openrouter/google/gemini-3-flash-preview")

report = sf.evaluate(candidate, benchmark=BENCHMARK_ID, limit=1)

HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efeff1;--sf-ink:#16181d;-…

## 4 · Report

The Report renders score, pass rate, coverage, cost and tokens, with every Case and the
Judge's per-criterion reasoning underneath. **&darr; report.json** downloads the portable
artifact — the same value `report.to_json()` returns.

In [6]:
report

Report(benchmark='draco/smoke', candidates=['gemini-3-flash-preview'], ok=True)

## 5 · Publish and retrieve

Publication accepts the evaluated `CandidateResult` directly. It derives the Benchmark id,
compiled URL4, models, accuracy counts, timestamps, and idempotency key from that immutable
result. Publication is independently opt-in so **Run All** never changes the Scoreboard.

The local Scoreboard accepts writes without login. Hosted deployments may require an
edge-verified identity or keep score submission closed.

In [7]:
PUBLISH_RESULT = True

submission = sf.leaderboards.submit(report.candidates.only) if PUBLISH_RESULT else None
submission if submission is not None else ("Set PUBLISH_RESULT = True to publish this result.")

LeaderboardScore(id=UUID('bd9ef405-6002-4f3b-86a4-7e3920909cb9'), version=1, benchmark_id='draco/smoke', spec_id='gemini-3-flash-preview', url4='(candidate:0.0:\'(model_1:0.0:/openrouter/google/gemini-3-flash-preview?max_tokens=4096&q=($input)!\\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\\')!\\\'$model_1\\\'\', (rows:0.0:(selected_rows:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/cases*(graded:0.0:(criteria:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/tasks((candidate_invocation:0.0:/benchmarks/candidate?web_search=true&web_search_exclude=alphaxiv.org:arxiv.org:huggingface.co:openrouter.ai:paperswithcode.com:research.perplexity.ai:semanticscholar.org&q=($item.input)!\'$candidate\')!\'$candidate_invocation\')!\'$item.id\'*(evaluated:0.0:(case_record:0.0:$item.case_record, check_record:0.0:$item.check_record, verdict_1:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/criterion-verdict(/openrouter/google/gemini-3.1-pro-p

In [8]:
published_score = sf.leaderboards.get_score(submission.id) if submission is not None else None
published_score

LeaderboardScore(id=UUID('bd9ef405-6002-4f3b-86a4-7e3920909cb9'), version=1, benchmark_id='draco/smoke', spec_id='gemini-3-flash-preview', url4='(candidate:0.0:\'(model_1:0.0:/openrouter/google/gemini-3-flash-preview?max_tokens=4096&q=($input)!\\\'Answer the request accurately and completely. Follow every instruction and formatting constraint in the request.\\\')!\\\'$model_1\\\'\', (rows:0.0:(selected_rows:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/cases*(graded:0.0:(criteria:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/tasks((candidate_invocation:0.0:/benchmarks/candidate?web_search=true&web_search_exclude=alphaxiv.org:arxiv.org:huggingface.co:openrouter.ai:paperswithcode.com:research.perplexity.ai:semanticscholar.org&q=($item.input)!\'$candidate\')!\'$candidate_invocation\')!\'$item.id\'*(evaluated:0.0:(case_record:0.0:$item.case_record, check_record:0.0:$item.check_record, verdict_1:0.0:/benchmarks/draco/smoke/7288a6646eeec3ca/criterion-verdict(/openrouter/google/gemini-3.1-pro-p

In [9]:
updated_leaderboard = (
    sf.leaderboards.get(BENCHMARK_ID, top=10) if submission is not None else leaderboard
)
updated_leaderboard

Leaderboard('draco/smoke', entries=1, baselines=0)

## 6 · Fork or replay the submitted URL4

`published_score.url4` is the raw evaluation expression stored by the Scoreboard. Its
`.to_python()` method returns an editable Model/Fusion and evaluation cell without spending.
Passing the URL4 itself to `sf.evaluate(...)` instead executes that exact, already
Benchmark-linked expression and returns a normal `Report`; do not pass `benchmark=` or `limit=`
again.

Replay is a fresh paid Evaluation and model output may differ, so it has its own opt-in guard.

In [10]:
fork_python = published_score.url4.to_python() if published_score is not None else None
print(fork_python) if fork_python is not None else "Publish a score to generate its Python fork."

import screamingface as sf

candidate = sf.Model(
    'openrouter/google/gemini-3-flash-preview',
)

report = sf.evaluate(
    candidate,
    benchmark='draco/smoke',
    limit=1,
)


In [11]:
REPLAY_RESULT = True

replayed_report = (
    sf.evaluate(published_score.url4) if REPLAY_RESULT and published_score is not None else None
)
replayed_report if replayed_report is not None else (
    "Set REPLAY_RESULT = True after publishing to run the stored URL4 again."
)

HTML(value='<style>\n.sf-ui{\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efeff1;--sf-ink:#16181d;-…

Report(benchmark='draco/smoke', candidates=['gemini-3-flash-preview'], ok=True)